# Grok-multimodal · FS11-FS12 Generation

Class-conditioned diffusion image gen + conditional short video gen.


In [ ]:
import os, json, math, random, time
from pathlib import Path
import numpy as np
os.environ.pop("CUDA_VISIBLE_DEVICES", None)
import torch, torch.nn as nn, torch.nn.functional as F
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
OUT=Path("/kaggle/working"); FIG=OUT/"figures"; RES=OUT/"results"
FIG.mkdir(parents=True, exist_ok=True); RES.mkdir(parents=True, exist_ok=True)
device=torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("device",device,"gpus",torch.cuda.device_count() if torch.cuda.is_available() else 0)
PROGRESS={}

def make_shape_image(kind, size=32):
    img=np.ones((size,size,3),np.float32)*0.95
    yy,xx=np.mgrid[0:size,0:size]; cy,cx=size//2,size//2
    if kind=="red_circle":
        m=(yy-cy)**2+(xx-cx)**2<=(size*0.28)**2; img[m]=(0.9,0.15,0.12)
    elif kind=="blue_square":
        m=(np.abs(yy-cy)<size*0.25)&(np.abs(xx-cx)<size*0.25); img[m]=(0.15,0.25,0.85)
    elif kind=="green_triangle":
        m=(yy>cy-size*0.25)&(yy<cy+size*0.3)
        m&=np.abs(xx-cx)<(yy-(cy-size*0.25))*0.7; img[m]=(0.15,0.75,0.25)
    else:
        raise ValueError(kind)
    return img


## FS11 · Image-text to image (mini DDPM)


In [ ]:
# Conditional image generation: tiny DDPM-style denoiser conditioned on class/text id
# Train on 32x32 synthetic shapes; condition = class embedding

CLASSES=["red_circle","blue_square","green_triangle"]
c2i={c:i for i,c in enumerate(CLASSES)}

def sample_xy(n=128):
    xs=[]; ys=[]
    for _ in range(n):
        k=random.choice(CLASSES)
        img=make_shape_image(k,32)
        img=np.clip(img+0.02*np.random.randn(*img.shape).astype(np.float32),0,1)
        xs.append(img.transpose(2,0,1)); ys.append(c2i[k])
    return torch.tensor(np.stack(xs),dtype=torch.float32), torch.tensor(ys)

# cosine schedule
T=50
betas=torch.linspace(1e-4, 0.05, T, device=device)
alphas=1.0-betas
alphabar=torch.cumprod(alphas,0)

def q_sample(x0, t, noise=None):
    if noise is None: noise=torch.randn_like(x0)
    ab=alphabar[t].view(-1,1,1,1)
    return ab.sqrt()*x0+(1-ab).sqrt()*noise, noise

class TinyUNet(nn.Module):
    def __init__(self, n_cls=3, base=32):
        super().__init__()
        self.emb=nn.Embedding(n_cls, base)
        self.tproj=nn.Linear(1, base)
        self.in_c=nn.Conv2d(3, base, 3, padding=1)
        self.d1=nn.Sequential(nn.Conv2d(base,base,3,padding=1),nn.ReLU(),nn.MaxPool2d(2))
        self.d2=nn.Sequential(nn.Conv2d(base,base*2,3,padding=1),nn.ReLU(),nn.MaxPool2d(2))
        self.mid=nn.Sequential(nn.Conv2d(base*2,base*2,3,padding=1),nn.ReLU())
        self.u1=nn.Sequential(nn.ConvTranspose2d(base*2,base,4,2,1),nn.ReLU())
        self.u2=nn.Sequential(nn.ConvTranspose2d(base*2,base,4,2,1),nn.ReLU())
        self.out=nn.Conv2d(base,3,3,padding=1)
    def forward(self,x,t,y):
        # x [B,3,H,W], t [B], y [B]
        B=x.size(0)
        te=self.tproj((t.float()/T).view(B,1)).view(B,-1,1,1)
        ye=self.emb(y).view(B,-1,1,1)
        h=self.in_c(x)+te+ye
        h1=self.d1(h); h2=self.d2(h1); m=self.mid(h2)
        u=self.u1(m); u=torch.cat([u,h1],1); u=self.u2(u)
        return self.out(u)

den=TinyUNet().to(device)
opt=torch.optim.Adam(den.parameters(), lr=2e-3)
hist11=[]
for epoch in range(1,31):
    den.train(); losses=[]
    for _ in range(40):
        x0,y=sample_xy(64); x0,y=x0.to(device),y.to(device)
        t=torch.randint(0,T,(x0.size(0),),device=device)
        xt,noise=q_sample(x0,t)
        pred=den(xt,t,y)
        loss=F.mse_loss(pred, noise)
        opt.zero_grad(set_to_none=True); loss.backward(); opt.step(); losses.append(loss.item())
    row={"epoch":epoch,"loss":round(float(np.mean(losses)),5)}
    hist11.append(row)
    if epoch%5==0: print(row)

@torch.no_grad()
def p_sample_loop(y, steps=T):
    den.eval()
    x=torch.randn(1,3,32,32,device=device)
    y=torch.tensor([y],device=device)
    for ti in reversed(range(steps)):
        t=torch.tensor([ti],device=device)
        eps=den(x,t,y)
        ab=alphabar[ti]; a=alphas[ti]; b=betas[ti]
        x=(1/a.sqrt())*(x - (b/(1-ab).sqrt())*eps)
        if ti>0:
            x=x+b.sqrt()*torch.randn_like(x)
    return x.clamp(0,1)[0].cpu().numpy().transpose(1,2,0)

gens=[]; fig,axes=plt.subplots(2,3,figsize=(8,5))
for j,k in enumerate(CLASSES):
    axes[0,j].imshow(make_shape_image(k,32)); axes[0,j].set_title("GT "+k,fontsize=8); axes[0,j].axis("off")
    g=p_sample_loop(c2i[k]); gens.append(g)
    axes[1,j].imshow(np.clip(g,0,1)); axes[1,j].set_title("gen "+k,fontsize=8); axes[1,j].axis("off")
fig.suptitle("FS11 text/class-conditioned diffusion (mini DDPM)")
fig.tight_layout(); fig.savefig(FIG/"fs11_gen.png",dpi=120); plt.close()

# crude quality: mean color distance to GT class prototype
def color_dist(gen, kind):
    gt=make_shape_image(kind,32)
    return float(np.linalg.norm(gen.mean(axis=(0,1))-gt.mean(axis=(0,1))))
qd={k:color_dist(gens[i],k) for i,k in enumerate(CLASSES)}
fs11={"stage":"FS11","method":"class-conditioned mini DDPM image generation",
      "history":hist11,"color_l2_to_gt_mean":qd,
      "vs_prev":"earlier stages understand/describe images; FS11 synthesizes pixels from condition",
      "figure":"figures/fs11_gen.png"}
(RES/"fs11.json").write_text(json.dumps(fs11,indent=2)); PROGRESS["FS11"]="ok"; print("FS11 DONE",qd)


## FS12 · Image-text to video


In [ ]:
# Image+text -> short video: generate frame sequence with motion prior conditioned on class
# Approach: predict residual sequence from noise with temporal conv conditioned on label
# Or simpler: train model to map (z, class) -> T frames of moving shape (supervised)

class VideoGen(nn.Module):
    def __init__(self, T=8, size=32, n_cls=3, z=16):
        super().__init__()
        self.T=T; self.size=size
        self.emb=nn.Embedding(n_cls, 32)
        self.fc=nn.Sequential(nn.Linear(z+32, 256), nn.ReLU(), nn.Linear(256, T*3*size*size))
    def forward(self, z, y):
        h=torch.cat([z, self.emb(y)], -1)
        return self.fc(h).view(z.size(0), self.T, 3, self.size, self.size)

def make_video(kind, T=8, size=32):
    frames=[]
    for t in range(T):
        img=np.ones((size,size,3),np.float32)*0.95
        yy,xx=np.mgrid[0:size,0:size]
        if kind=="red_circle":
            cx=int(size*(0.25+0.5*t/(T-1))); cy=size//2; r=size*0.16
            m=(yy-cy)**2+(xx-cx)**2<=r**2; img[m]=(0.9,0.15,0.12)
        elif kind=="blue_square":
            cy=int(size*(0.25+0.5*t/(T-1))); cx=size//2; s=int(size*0.16)
            m=(np.abs(yy-cy)<s)&(np.abs(xx-cx)<s); img[m]=(0.15,0.25,0.85)
        else:
            cy,cx=size//2,size//2
            sc=0.12+0.08*(t/(T-1))
            m=(yy>cy-size*sc)&(yy<cy+size*sc*1.2)
            m&=np.abs(xx-cx)<(yy-(cy-size*sc))*0.8; img[m]=(0.15,0.75,0.25)
        frames.append(img.transpose(2,0,1))
    return np.stack(frames)

vg=VideoGen().to(device)
opt=torch.optim.Adam(vg.parameters(), lr=2e-3)
hist12=[]
for epoch in range(1,41):
    vg.train(); losses=[]
    for _ in range(30):
        ys=[]; vs=[]
        for _b in range(32):
            k=random.choice(CLASSES); ys.append(c2i[k]); vs.append(make_video(k))
        y=torch.tensor(ys,device=device)
        v=torch.tensor(np.stack(vs),dtype=torch.float32,device=device)
        z=torch.randn(len(ys),16,device=device)
        pred=vg(z,y)
        loss=F.mse_loss(pred,v)
        opt.zero_grad(set_to_none=True); loss.backward(); opt.step(); losses.append(loss.item())
    row={"epoch":epoch,"mse":round(float(np.mean(losses)),5)}
    hist12.append(row)
    if epoch%10==0: print(row)

@torch.no_grad()
def gen_video(kind):
    vg.eval(); z=torch.randn(1,16,device=device); y=torch.tensor([c2i[kind]],device=device)
    v=vg(z,y)[0].cpu().numpy()  # T,3,H,W
    return np.clip(v.transpose(0,2,3,1),0,1)

fig,axes=plt.subplots(3,8,figsize=(12,4.5))
mses={}
for i,k in enumerate(CLASSES):
    gt=make_video(k).transpose(0,2,3,1)
    gen=gen_video(k)
    mses[k]=float(np.mean((gt-gen)**2))
    for t in range(8):
        # show gen
        axes[i,t].imshow(gen[t]); axes[i,t].axis("off")
        if t==0: axes[i,t].set_ylabel(k+" gen",fontsize=7)
fig.suptitle("FS12 image/text-cond video frames (moving shapes)")
fig.tight_layout(); fig.savefig(FIG/"fs12_video_gen.png",dpi=120); plt.close()
fs12={"stage":"FS12","method":"conditional latent MLP video generator (supervised motion)",
      "history":hist12,"frame_mse":mses,
      "vs_prev":"FS11 single image; FS12 emits temporal sequence conditioned on class/text id",
      "figure":"figures/fs12_video_gen.png"}
(RES/"fs12.json").write_text(json.dumps(fs12,indent=2)); PROGRESS["FS12"]="ok"; print("FS12 DONE",mses)


In [ ]:
summary={"notebook":"Grok-multimodal-fs11-fs12-generation","progress":PROGRESS,"device":str(device)}
(RES/"summary_fs11_fs12.json").write_text(json.dumps(summary,indent=2))
(OUT/"SUCCESS").write_text("ok\n"); print(summary)
